In [ ]:
%pip install git+https://github.com/Eickland/HumSpectra.git --upgrade

In [1]:
import os
import glob
from pathlib import Path
import pandas as pd
import numpy as np
import tensorly as tl
from tensorly.decomposition import parafac
from tensorly.decomposition import non_negative_parafac, non_negative_parafac_hals
import matplotlib.pyplot as plt
from scipy import stats
from mpl_toolkits.axes_grid1 import make_axes_locatable

import HumSpectra.fluorescence as fl
from HumSpectra.parafac import EEMDataLoader, OpticalDataAnalyzer, ComponentVisualizer

In [2]:
wdir_2024 = Path.cwd().parent / "initial data" / "2024"
fluo_paths = Path.joinpath(wdir_2024, "Fluorescence")

In [3]:
loader = EEMDataLoader(fluo_paths)
eem_tensor,em_waves,ex_waves,sp_names = loader.load_eem_data(index_col=0)
data_info = loader.get_data_info()

Создан тензор размерности: (188, 70, 40)
Образцы: 188
Длины волн испускания: 70
Длины волн возбуждения: 40


In [4]:
cleaned, mask, residuals, info = loader.robust_detect_outliers(
        eem_tensor, rank=4, threshold_std=3.0, verbose=True
    )

Обнаружено выбросов: 11965


In [5]:
print(f"Процент выбросов: {info['n_outliers'] / eem_tensor.size * 100:.2f}%")
print(f"Обнаружено выбросов: {info['n_outliers']}")

Процент выбросов: 2.27%
Обнаружено выбросов: 11965


In [6]:
analyzer = ComponentVisualizer()
factors = analyzer.fit_parafac(
    cleaned,
    excitation_wavelengths=ex_waves,
    emission_wavelengths=em_waves,
    sample_names=sp_names)


=== ВАЛИДАЦИЯ РЕЗУЛЬТАТОВ ===
Количество факторов: 3
✅ Фактор 0: неотрицательный (min = 0.00e+00)
✅ Фактор 1: неотрицательный (min = 0.00e+00)
✅ Фактор 2: неотрицательный (min = 0.00e+00)
Ошибка реконструкции: 8.68e-02
Сходимость: 1.12e-07
Фактор 0: форма (188, 3), диапазон [0.000, 0.662]
Фактор 1: форма (70, 3), диапазон [0.000, 1.237]
Фактор 2: форма (40, 3), диапазон [0.000, 1.568]

=== МЕТРИКИ КАЧЕСТВА ===
Объясненная дисперсия: 99.25%
Core Consistency: 99.56%


In [7]:
analyzer.export_loadings_to_csv(Path.joinpath(Path.cwd().parent,"Server","Received data","Parafac","parafac_2024.csv"))

Таблица сохранена в c:\Users\mnbv2\Desktop\lab\KNP analysis\Server\Received data\Parafac\parafac_2024.csv
